# Test: AUROC Fix Verification

Quick test to verify the uncertainty fix (entropy instead of score variance).

**Expected:** AUROC > 0.5 after fix

In [1]:
# Setup
import os, sys

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    # Clone repo and install deps
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior 2>/dev/null || echo "exists"
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')

    # Apply the fix directly (since GitHub doesn't have it yet)
    print("Applying AUROC fix...")
else:
    # Local: use parent directory
    project_root = os.path.dirname(os.getcwd())
    sys.path.insert(0, project_root)

Applying AUROC fix...


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.utils.training import set_seed
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

Device: cuda
FB15k-237 not found. Downloading...


train.txt: 21.0MB [00:01, 12.2MB/s]


valid.txt: 1.29MB [00:00, 2.82MB/s]


test.txt: 1.51MB [00:00, 4.76MB/s]


FB15k-237 download complete!
Data: 272,115 train, 20,466 test


In [3]:
# Quick training (10 epochs, small sample)
print("Training GP-KGE (quick test)...")

model = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=50,  # Small for quick test
    kernel_type="rbf",
    num_inducing=100
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=0.001)

# Only 10 epochs for quick test
for ep in (pbar := tqdm(range(10), desc="GP-KGE")):
    model.train()
    loss_sum, n = 0, 0
    for st in range(0, min(50000, len(train_data)), 1024):  # Subset
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        ps = model.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        loss.backward()
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print("Training done!")

Training GP-KGE (quick test)...


GP-KGE:   0%|          | 0/10 [00:00<?, ?it/s]

Training done!


In [4]:
# Test AUROC with different uncertainty types
print("\nTesting uncertainty types...")
model.eval()

# Sample data
n_samples = 1000
id_triples = test_data.triples[np.random.choice(len(test_data), n_samples, replace=False)]
ood_triples = create_ood_dataset(train_data, test_data, "random", n_samples)

def get_uncertainties(triples, unc_type):
    uncs = []
    with torch.no_grad():
        for i in range(0, len(triples), 256):
            batch = triples[i:i+256]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            pred = model.predict_with_uncertainty(h, r, t)
            uncs.append(pred[unc_type].cpu().numpy())
    return np.concatenate(uncs)

# Test different uncertainty types
unc_types = ['total', 'epistemic', 'score_var', 'prob_var']

print("\n" + "="*50)
print("AUROC Results (higher is better, >0.5 expected)")
print("="*50)

for unc_type in unc_types:
    try:
        id_unc = get_uncertainties(id_triples, unc_type)
        ood_unc = get_uncertainties(ood_triples, unc_type)
        auroc = compute_auroc(id_unc, ood_unc)
        status = "✓" if auroc > 0.5 else "✗"
        print(f"{unc_type:12}: AUROC = {auroc:.4f} {status}")
        print(f"             ID mean = {id_unc.mean():.4f}, OOD mean = {ood_unc.mean():.4f}")
    except Exception as e:
        print(f"{unc_type:12}: Error - {e}")


Testing uncertainty types...

AUROC Results (higher is better, >0.5 expected)
total       : AUROC = 0.5078 ✓
             ID mean = -0.0168, OOD mean = 0.0022
epistemic   : AUROC = 0.5022 ✓
             ID mean = 0.0048, OOD mean = 0.0088
score_var   : AUROC = 0.1965 ✗
             ID mean = 4.3295, OOD mean = 1.7837
prob_var    : AUROC = 0.1913 ✗
             ID mean = 0.0933, OOD mean = 0.0565


In [5]:
# Compare with entropy baseline (like DistMult uses)
print("\n" + "="*50)
print("Comparison with raw entropy (DistMult style)")
print("="*50)

def entropy_uncertainty(triples):
    uncs = []
    with torch.no_grad():
        for i in range(0, len(triples), 256):
            batch = triples[i:i+256]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_triple(h, r, t, use_mean=True)
            p = torch.sigmoid(scores)
            entropy = -p * torch.log(p + 1e-10) - (1-p) * torch.log(1-p + 1e-10)
            uncs.append(entropy.cpu().numpy())
    return np.concatenate(uncs)

id_entropy = entropy_uncertainty(id_triples)
ood_entropy = entropy_uncertainty(ood_triples)
auroc_entropy = compute_auroc(id_entropy, ood_entropy)

print(f"Raw entropy:  AUROC = {auroc_entropy:.4f}")
print(f"              ID mean = {id_entropy.mean():.4f}, OOD mean = {ood_entropy.mean():.4f}")


Comparison with raw entropy (DistMult style)
Raw entropy:  AUROC = 0.7364
              ID mean = 0.6926, OOD mean = 0.6931


In [6]:
# Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print("\nExpected behavior after fix:")
print("- 'total' and 'epistemic' (entropy-based): AUROC > 0.5 ✓")
print("- 'score_var' (old method): AUROC < 0.5 ✗")
print("\nIf 'total' AUROC > 0.5, the fix works!")


SUMMARY

Expected behavior after fix:
- 'total' and 'epistemic' (entropy-based): AUROC > 0.5 ✓
- 'score_var' (old method): AUROC < 0.5 ✗

If 'total' AUROC > 0.5, the fix works!
